In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE customers (
    customer_id INTEGER,
    customer_name TEXT,
    city TEXT
)
""")

print("Table created successfully!")

Table created successfully!


In [5]:
cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER,
    customer_id INTEGER,
    order_date TEXT,
    revenue REAL
)
""")

In [6]:
cursor.execute("""
CREATE TABLE products (
    product_id INTEGER,
    product_name TEXT,
    category TEXT
)
""")

In [7]:
cursor.execute("""
CREATE TABLE order_items (
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER
)
""")

In [8]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print(cursor.fetchall())

[('customers',), ('orders',), ('products',), ('order_items',)]


In [9]:
cursor.executemany("""
INSERT INTO customers VALUES (?, ?, ?)
""", [
    (101, 'Amit', 'Mumbai'),
    (102, 'Neha', 'Pune'),
    (103, 'Rahul', 'Delhi'),
    (104, 'Priya', 'Mumbai'),
    (105, 'Karan', 'Pune')
])

In [10]:
cursor.executemany("""
INSERT INTO orders VALUES (?, ?, ?, ?)
""", [
    (1, 101, '2025-01-05', 5000),
    (2, 102, '2025-01-08', 7000),
    (3, 101, '2025-02-01', 3000),
    (4, 103, '2025-02-10', 9000),
    (5, 102, '2025-03-01', 2000),
    (6, 104, '2025-03-12', 4000),
    (7, 105, '2025-03-15', 6000),
    (8, 103, '2025-04-01', 5000)
])

In [11]:
cursor.executemany("""
INSERT INTO products VALUES (?, ?, ?)
""", [
    (1, 'Laptop', 'Electronics'),
    (2, 'Mouse', 'Electronics'),
    (3, 'Keyboard', 'Electronics'),
    (4, 'Chair', 'Furniture'),
    (5, 'Desk', 'Furniture')
])

In [12]:
cursor.executemany("""
INSERT INTO order_items VALUES (?, ?, ?)
""", [
    (1, 1, 1),
    (1, 2, 2),
    (2, 1, 1),
    (3, 3, 1),
    (4, 5, 1),
    (5, 2, 3),
    (6, 4, 1),
    (7, 1, 1),
    (8, 5, 2)
])

conn.commit()

## Data Preview

In [13]:
pd.read_sql_query("SELECT * FROM customers", conn)

,customer_id,customer_name,city
0,101,Amit,Mumbai
1,102,Neha,Pune
2,103,Rahul,Delhi
3,104,Priya,Mumbai
4,105,Karan,Pune


In [14]:
pd.read_sql_query("SELECT * FROM orders", conn)

,order_id,customer_id,order_date,revenue
0,1,101,2025-01-05,5000.0
1,2,102,2025-01-08,7000.0
2,3,101,2025-02-01,3000.0
3,4,103,2025-02-10,9000.0
4,5,102,2025-03-01,2000.0
5,6,104,2025-03-12,4000.0
6,7,105,2025-03-15,6000.0
7,8,103,2025-04-01,5000.0


In [15]:
pd.read_sql_query("SELECT * FROM products", conn)

,product_id,product_name,category
0,1,Laptop,Electronics
1,2,Mouse,Electronics
2,3,Keyboard,Electronics
3,4,Chair,Furniture
4,5,Desk,Furniture


In [16]:
pd.read_sql_query("SELECT * FROM order_items", conn)

,order_id,product_id,quantity
0,1,1,1
1,1,2,2
2,2,1,1
3,3,3,1
4,4,5,1
5,5,2,3
6,6,4,1
7,7,1,1
8,8,5,2


## Total Revenue

In [17]:
query = """
SELECT SUM(revenue) AS total_revenue
FROM orders
"""
pd.read_sql_query(query, conn)

,total_revenue
0,41000.0


## Top Customers

In [18]:
query = """
SELECT customer_id,
       SUM(revenue) AS total_revenue
FROM orders
GROUP BY customer_id
ORDER BY total_revenue DESC
LIMIT 1
"""
pd.read_sql_query(query, conn)

,customer_id,total_revenue
0,103,14000.0


## Revenue by City

In [19]:
query = """
SELECT c.city,
       SUM(o.revenue) AS total_revenue
FROM customers c
INNER JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.city
ORDER BY total_revenue DESC
"""
pd.read_sql_query(query, conn)

,city,total_revenue
0,Pune,15000.0
1,Delhi,14000.0
2,Mumbai,12000.0


## Repeat Customers

In [20]:
query = """
SELECT customer_id,
       COUNT(order_id) AS order_count
FROM orders
GROUP BY customer_id
HAVING COUNT(order_id) > 1
"""
pd.read_sql_query(query, conn)

,customer_id,order_count
0,101,2
1,102,2
2,103,2


## Average Order Value

In [21]:
query = """
SELECT AVG(revenue) AS avg_order_value
FROM orders
"""
pd.read_sql_query(query, conn)

,avg_order_value
0,5125.0


In [22]:
df = pd.read_sql_query(
    """
    SELECT c.city,
           SUM(o.revenue) AS total_revenue
    FROM customers c
    INNER JOIN orders o
    ON c.customer_id = o.customer_id
    GROUP BY c.city
    """,
    conn
)

In [23]:
df

,city,total_revenue
0,Delhi,14000.0
1,Mumbai,12000.0
2,Pune,15000.0


# Pandas Analysis

In [24]:
city_df = pd.read_sql_query("""
SELECT c.city,
       SUM(o.revenue) AS total_revenue
FROM customers c
INNER JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.city
""", conn)

city_df

,city,total_revenue
0,Delhi,14000.0
1,Mumbai,12000.0
2,Pune,15000.0


In [25]:
city_df.sort_values(
    by="total_revenue",
    ascending=False
)

,city,total_revenue
2,Pune,15000.0
0,Delhi,14000.0
1,Mumbai,12000.0


## Bussiness Insights

1. Total company revenue is ₹41,000.

2. Customer 103 is the highest-value customer with ₹14,000 revenue.

3. Pune is the highest-performing city with ₹15,000 revenue.

4. Three customers are repeat customers.

5. Average Order Value is ₹5,125.

## Executive Summary

This ecommerce analysis identified key revenue drivers and customer behavior patterns.

Key Findings:
- Revenue: ₹41,000
- Top Customer: Customer 103
- Top City: Pune
- Repeat Customers: 3
- Average Order Value: ₹5,125

Recommendations:
- Focus marketing efforts in Pune.
- Retain high-value customers.
- Increase repeat purchases from one-time customers.

## Project Documentation
Project Overview

This project analyzes ecommerce sales data using SQLite, SQL, Pandas, and Jupyter Notebook.

Objectives:
- Analyze revenue performance
- Identify top customers
- Analyze city performance
- Measure repeat customer behavior
- Generate business recommendations

## Data Preview

In [26]:
pd.read_sql_query("SELECT * FROM customers", conn)

,customer_id,customer_name,city
0,101,Amit,Mumbai
1,102,Neha,Pune
2,103,Rahul,Delhi
3,104,Priya,Mumbai
4,105,Karan,Pune


In [27]:
pd.read_sql_query("SELECT * FROM orders", conn)

,order_id,customer_id,order_date,revenue
0,1,101,2025-01-05,5000.0
1,2,102,2025-01-08,7000.0
2,3,101,2025-02-01,3000.0
3,4,103,2025-02-10,9000.0
4,5,102,2025-03-01,2000.0
5,6,104,2025-03-12,4000.0
6,7,105,2025-03-15,6000.0
7,8,103,2025-04-01,5000.0
